In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

# Get the current user's home directory
HOME_DIR = Path.home()

# Define model storage directory
FILES_DIR = HOME_DIR / ".files"
MODEL_DIR = FILES_DIR / "models"

# Create directories if they do not exist
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Define model path
MODEL_PATH = MODEL_DIR / "yolo11n.pt"

# Download model only if it does not already exist
MODEL_URL = (
    "https://github.com/ultralytics/assets/"
    "releases/download/v8.3.0/yolo11n.pt"
)

if not MODEL_PATH.exists():
    print("YOLO11n model not found. Downloading...")

    urlretrieve(MODEL_URL, MODEL_PATH)

    print("Model downloaded successfully!")

else:
    print("YOLO11n model already exists. Skipping download.")

print("Model location:", MODEL_PATH)

In [2]:
import cv2
import matplotlib.pyplot as plt

from ultralytics import YOLO

In [ ]:
PROJECT_DIR = Path.cwd()

DATASET_DIR = PROJECT_DIR / "dataset"
IMAGE_DIR = DATASET_DIR / "images"
VIDEO_DIR = DATASET_DIR

OUTPUT_DIR = PROJECT_DIR / "runs" / "person_detection"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Images:", IMAGE_DIR)
print("Outputs:", OUTPUT_DIR)

In [ ]:
model = YOLO(str(MODEL_PATH))

print("YOLO11n model loaded successfully!")

In [5]:
CONFIDENCE_THRESHOLD = 0.25
IMAGE_SIZE = 640
PERSON_CLASS_ID = 0
DEVICE = "cpu"

In [ ]:
image_path = IMAGE_DIR / "construction_worker.jpg"

if not image_path.exists():
    raise FileNotFoundError(f"Image not found: {image_path}")

print("Input Image:", image_path)

In [ ]:
results = model.predict(
    source=str(image_path),
    classes=[PERSON_CLASS_ID],
    conf=CONFIDENCE_THRESHOLD,
    imgsz=IMAGE_SIZE,
    device=DEVICE
)

print("Person detection completed!")

In [ ]:
for result in results:
    for index, box in enumerate(result.boxes):

        x1, y1, x2, y2 = box.xyxy[0].tolist()

        width = x2 - x1
        height = y2 - y1

        center_x = (x1 + x2) / 2
        center_y = (y1 + y2) / 2

        confidence = box.conf[0].item()
        class_id = int(box.cls[0].item())

        print(f"Person {index + 1}")
        print(f"Class ID: {class_id}")
        print(f"Confidence: {confidence:.4f}")
        print(f"X1: {x1:.2f}, Y1: {y1:.2f}")
        print(f"X2: {x2:.2f}, Y2: {y2:.2f}")
        print(f"Width: {width:.2f}")
        print(f"Height: {height:.2f}")
        print(f"Center X: {center_x:.2f}")
        print(f"Center Y: {center_y:.2f}")
        print("-" * 40)

In [ ]:
result_image = results[0].plot()

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(result_image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("YOLO11n Person Detection")
plt.show()

In [ ]:
output_path = OUTPUT_DIR / image_path.name

cv2.imwrite(str(output_path), result_image)

print("Detection output saved:", output_path)